# Двор, карточки Pokemon и накопление преимущества

Более реалистичная формулировка модели:

> У детей есть случайные стартовые коллекции карточек разной редкости, типа и рыночной ценности. У каждого ребенка свои субъективные предпочтения: кому-то нужны огненные карты, кому-то редкие, кому-то конкретный тип для воображаемой колоды. Чем больше и разнообразнее коллекция, тем больше независимых возможностей обмена она создает: больше карточек можно предложить, больше детей могут захотеть что-то из твоей коллекции, и проще собрать bundle из нескольких менее нужных карт ради одной более нужной.
>
> Сделка происходит только если обе стороны субъективно выигрывают. При этом владелец ликвидной карты не обязан принимать рыночный минус: если к нему приходят за нужной картой, он может запросить bundle с небольшим рыночным premium. Покупатель может согласиться, потому что субъективно эта карта для него ценнее, чем отданный набор. Даже при таких добровольных обменах рыночная ценность коллекций может концентрироваться у тех, кто случайно получил лучшую стартовую позицию: у них больше leads, больше ликвидности и больше способов подобрать сделку.

Что важно: модель не говорит, что дети с большим стартовым запасом “умнее” или “лучше торгуются”. Преимущество задается не качеством одной сделки, а числом возможностей и ликвидностью портфеля. Мы отдельно считаем субъективную полезность и рыночную ценность: utility может расти у обеих сторон сделки, а market value при этом перераспределяется неравномерно.

**Gini** - индекс неравенства распределения. `0` означает полное равенство: у всех одинаковая ценность коллекции. `1` означает предельную концентрацию: почти вся ценность у одного ребенка.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pokemon_yard_model import (
    make_initial_distribution,
    simulate_exchanges,
    describe_distribution,
    group_shares,
    lorenz_curve,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

## 1. Старт: количество карточек и рыночная ценность

Сначала задаем физическое число карточек у каждого ребенка нормальным распределением с отсечением в ноль. Затем каждая отдельная карточка получает тип, редкость и рыночную ценность. Поэтому два ребенка с похожим числом карточек могут иметь очень разную стартовую рыночную позицию.

In [ ]:
initial_card_counts = make_initial_distribution(
    n_children=100,
    mean_cards=20,
    std_cards=12,
    seed=7,
)

result = simulate_exchanges(
    initial_card_counts,
    steps=180,
    card_lead_probability=0.008,
    max_bundle_size=4,
    search_width=10,
    min_utility_gain=0.12,
    seed=11,
)

initial_value = result.initial
final_value = result.history[-1]
initial_utility = result.utility_history[0]
final_utility = result.utility_history[-1]
final_card_counts = result.card_count_history[-1]
order = result.initial_order
colors = np.array(["#d14a4a"] * 20 + ["#4c78a8"] * 60 + ["#3b9b67"] * 20)

pd.DataFrame({
    "initial_cards": pd.Series(describe_distribution(result.initial_card_counts)),
    "final_cards": pd.Series(describe_distribution(final_card_counts)),
    "initial_market_value": pd.Series(describe_distribution(initial_value)),
    "final_market_value": pd.Series(describe_distribution(final_value)),
    "initial_subjective_utility": pd.Series(describe_distribution(initial_utility)),
    "final_subjective_utility": pd.Series(describe_distribution(final_utility)),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(np.arange(len(initial_card_counts)), result.initial_card_counts[order], color=colors, width=0.9)
axes[0].set_title("Физическое число карточек, сортировка по стартовой рыночной ценности")
axes[0].set_xlabel("Ранг по стартовой рыночной ценности")
axes[0].set_ylabel("Карточки")

axes[1].bar(np.arange(len(initial_value)), initial_value[order], color=colors, width=0.9)
axes[1].set_title("Стартовая рыночная ценность коллекций")
axes[1].set_xlabel("Ранг по стартовой рыночной ценности")
axes[1].set_ylabel("Market value")

plt.tight_layout()

## 2. Добровольные обмены

На каждом шаге:

1. Каждая имеющаяся карточка независимо может создать торговый lead. Это означает, что больше карточек дает больше попыток обмена без предположения, что одна попытка у “богатого” качественно лучше.
2. У каждой карточки есть рыночная ценность, тип и редкость. У каждого ребенка есть субъективные предпочтения по типам.
3. Инициатор ищет bundle-сделку: например, отдать несколько менее нужных ему карт ради одной более нужной.
4. Сделка исполняется только если обе стороны получают положительный субъективный выигрыш.
5. Владелец ликвидной карты принимает сделку только если не теряет рыночную ценность: покупатель может переплатить market value, потому что получает большую субъективную utility.
6. Общее число карточек и общая рыночная ценность во дворе сохраняются, но распределение рыночной ценности между детьми меняется.

In [ ]:
total_leads = result.opportunity_history.sum(axis=0)
market_change = final_value - initial_value

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(result.initial_card_counts, total_leads, s=50, alpha=0.78, color="#4c78a8")
axes[0].set_title("Больше карточек -> больше торговых leads")
axes[0].set_xlabel("Карточки на старте")
axes[0].set_ylabel("Всего leads за симуляцию")

axes[1].scatter(initial_value, market_change, s=50, alpha=0.78, color="#d14a4a")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Изменение рыночной ценности против стартовой позиции")
axes[1].set_xlabel("Рыночная ценность на старте")
axes[1].set_ylabel("Финал минус старт")

plt.tight_layout()

In [ ]:
steps = np.arange(result.history.shape[0])
trade_steps = [event[0] for event in result.events]
trade_count_by_step = np.bincount(trade_steps, minlength=result.history.shape[0] - 1) if trade_steps else np.zeros(result.history.shape[0] - 1)
cumulative_trades = np.concatenate([[0], np.cumsum(trade_count_by_step)])

total_utility = result.utility_history.sum(axis=1)
market_gini = np.array([describe_distribution(row)["gini"] for row in result.history])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, total_utility, color="#3b9b67")
axes[0].set_title("Суммарная субъективная utility растет")
axes[0].set_xlabel("Шаг")
axes[0].set_ylabel("Total subjective utility")

axes[1].plot(steps, market_gini, color="black", label="Gini market value")
axes[1].plot(steps, cumulative_trades / max(cumulative_trades.max(), 1), color="#4c78a8", alpha=0.65, label="Сделки, нормировано")
axes[1].set_title("Добровольные сделки и концентрация ценности")
axes[1].set_xlabel("Шаг")
axes[1].set_ylabel("Значение")
axes[1].legend()

plt.tight_layout()

## 3. Перетекание рыночной ценности по стартовым группам

Делим детей не по финалу, а по стартовой рыночной позиции: нижние 20%, средние 60%, верхние 20%. Это показывает, как случайное начальное положение влияет на последующие возможности.

In [ ]:
shares = group_shares(result.history, initial_value)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, shares["bottom_20"] * 100, color="#d14a4a", label="нижние 20% старта")
axes[0].plot(steps, shares["middle_60"] * 100, color="#4c78a8", label="средние 60%")
axes[0].plot(steps, shares["top_20"] * 100, color="#3b9b67", label="верхние 20% старта")
axes[0].set_title("Доля всей рыночной ценности у стартовых групп")
axes[0].set_xlabel("Шаг")
axes[0].set_ylabel("Доля market value, %")
axes[0].legend()

card_shares = group_shares(result.card_count_history, result.initial_card_counts)
axes[1].plot(steps, card_shares["bottom_20"] * 100, color="#d14a4a", label="нижние 20% по числу карт")
axes[1].plot(steps, card_shares["middle_60"] * 100, color="#4c78a8", label="средние 60%")
axes[1].plot(steps, card_shares["top_20"] * 100, color="#3b9b67", label="верхние 20% по числу карт")
axes[1].set_title("Доля физического числа карточек у стартовых групп")
axes[1].set_xlabel("Шаг")
axes[1].set_ylabel("Доля карточек, %")
axes[1].legend()

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(np.arange(len(initial_value)), initial_value[order], color=colors, alpha=0.45, label="старт")
axes[0].bar(np.arange(len(final_value)), final_value[order], color=colors, alpha=0.9, width=0.55, label="финал")
axes[0].set_title("Те же дети, отсортированные по стартовой рыночной ценности")
axes[0].set_xlabel("Ранг на старте")
axes[0].set_ylabel("Market value")
axes[0].legend()

x0, y0 = lorenz_curve(initial_value)
x1, y1 = lorenz_curve(final_value)
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", label="равенство")
axes[1].plot(x0, y0, color="#4c78a8", label="старт")
axes[1].plot(x1, y1, color="#d14a4a", label="финал")
axes[1].set_title("Кривая Лоренца по рыночной ценности")
axes[1].set_xlabel("Доля детей, от бедных к богатым")
axes[1].set_ylabel("Доля market value")
axes[1].legend()

plt.tight_layout()

## 4. Manim: поток рыночной ценности

Эта сцена показывает детей в порядке стартовой рыночной позиции. Высота столбца - текущая рыночная ценность коллекции. Желтые точки показывают заметные net-потоки рыночной ценности внутри окна анимации.

В Binder важно запускать Manim без флага `-p`: этот флаг пытается открыть видео через системный viewer (`xdg-open`), которого в Binder обычно нет.

In [ ]:
from IPython.display import Video, display
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "manim", "-ql", "manim_card_flow.py", "CardFlowScene"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

video_path = "media/videos/manim_card_flow/480p15/CardFlowScene.mp4"
display(Video(video_path, embed=True, html_attributes="controls"))

## 5. Manim: круговая визуализация bundle-сделок

В этой версии дети расположены по кругу. Каждый ребенок - кружок; цвет показывает стартовую группу по рыночной ценности, а диаметр кружка пропорционален текущей рыночной ценности коллекции. Желтые точки показывают net-потоки рыночной ценности после добровольных bundle-сделок: один кружок уменьшается, другой увеличивается. Физическое число карточек остается дополнительной метрикой в подписи сцены.

In [ ]:
from IPython.display import Video, display
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "manim", "-ql", "manim_card_flow.py", "CircleTradeScene"],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

circle_video_path = "media/videos/manim_card_flow/480p15/CircleTradeScene.mp4"
display(Video(circle_video_path, embed=True, html_attributes="controls"))